In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

In [3]:
print("Iniciando o Setup e Limpeza do Schema...")

try:
    # Localizadondo a raiz do projeto
    current_path = Path(".").resolve()
    db_path = None

    # Procura o ficheiro na pasta atual e em todas as pastas acima
    for folder in [current_path] + list(current_path.parents):
        if (folder / "aposta_ganha_dw.db").exists():
            db_path = folder / "aposta_ganha_dw.db"
            break

    if not db_path:
        raise FileNotFoundError("Base de dados 'aposta_ganha_dw.db' não foi encontrada. Tem certeza que rodou a Semana 1 novamente?")

    print(f"Base de dados localizada com sucesso em: {db_path}")

    # 2. Conectando à Base de Dados local
    conn = sqlite3.connect(str(db_path))
    cursor = conn.cursor()

    # 3. Bloco SQL Intensivo (schema_setup.sql)
    cursor.executescript("""
        -- Tratamento de valores nulos
        UPDATE fato_transacoes SET valor_ganho = 0.0 WHERE valor_ganho IS NULL;
        UPDATE fato_transacoes SET valor_perdido = 0.0 WHERE valor_perdido IS NULL;

        -- Padronização dos textos                 
        UPDATE dim_usuarios SET preferencia_jogo = LOWER(preferencia_jogo);

        -- Criação da tabela
        DROP VIEW IF EXISTS vw_transacoes_limpas;
        
        CREATE VIEW vw_transacoes_limpas AS
        SELECT 
            t.transacao_id, t.user_id, u.nome, u.estado, u.preferencia_jogo, u.flag_vip,
            t.data_hora, DATE(t.data_hora) as data_transacao,
            t.tipo_transacao, t.valor_transacao, t.valor_ganho, t.valor_perdido
        FROM fato_transacoes t
        INNER JOIN dim_usuarios u ON t.user_id = u.user_id;
    """)

    conn.commit()
    print("Limpeza de nulos e padronização de Texto concluídas com sucesso!")
    print("Tabela 'vw_transacoes_limpas' criada!")

    # 4. Validação da Qualidade dos Dados
    print("Pré-visualização da Tabela:")
    df_view = pd.read_sql_query("SELECT * FROM vw_transacoes_limpas LIMIT 5;", conn)
    print()
    print(df_view.to_string(index=False))

    conn.close()

except Exception as e:
    print(f"Falha na execução do schema_setup: {e}")

Iniciando o Setup e Limpeza do Schema...
Base de dados localizada com sucesso em: C:\Users\ewert\Documents\Projetos\aposta-ganha-analytics-framework\aposta_ganha_dw.db
Limpeza de nulos e padronização de Texto concluídas com sucesso!
Tabela 'vw_transacoes_limpas' criada!
Pré-visualização da Tabela:

                        transacao_id  user_id                   nome estado preferencia_jogo  flag_vip           data_hora data_transacao   tipo_transacao  valor_transacao  valor_ganho  valor_perdido
714d901d-e858-46ff-aeae-1467911322f9   770487 João Guilherme da Cruz     ES         esportes         0 2026-05-26 00:00:00     2026-05-26 aposta_esportiva            14.08         0.00          14.08
15d92262-18ba-438f-9c8a-b5a040ec1632   770487 João Guilherme da Cruz     ES         esportes         0 2026-05-26 06:00:00     2026-05-26   aposta_cassino            90.30       198.26           0.00
5697a811-6506-486f-aa44-1b64b4621383   770487 João Guilherme da Cruz     ES         esportes        